# Momentum — Execution Scenario Analysis
Tests how portfolio OOS performance changes depending on when trades are filled.
Signals and trade logic are **never modified** — only the fill price changes.

Hourly data is fetched **once** — swap execution time in cell 2 without re-fetching.

---
### How the baseline works
The engine applies a 1-bar shift: **signal on day T → execute at Close[T]**.
Close[T] on daily Binance data = midnight UTC = ~1am UK time.

The scenarios below replace `Close[T]` on signal days only, leaving holding-period returns untouched.

| Scenario | Entry fills at | Exit fills at |
|---|---|---|
| `'close'` | Close[T] — midnight UTC (baseline) | Close[T] — midnight UTC |
| `integer hour` | HH:00 UTC on T+1 (e.g. `10` = 10am) | HH:00 UTC on T+1 |
| `'worst_long'` | High of T+1 — worst intraday entry | Low of T+1 — worst intraday exit |

In [ ]:
import os
import sys
import pandas as pd

# ── repo root — works on both Mac and Windows ─────────────────────────────────
ROOT   = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows

WF_DIR = os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'wf_testing')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos

client = get_binance_client()

---
## 1 — Load OOS results and fetch hourly data (run once)

In [ ]:
# ── load daily OOS pkl files ───────────────────────────────────────────────────
coin_dfs = {}
for fname in os.listdir(WF_DIR):
    if fname.endswith('_oos.pkl'):
        coin = fname.replace('_oos.pkl', '').upper()
        coin_dfs[coin] = pd.read_pickle(os.path.join(WF_DIR, fname))

print(f'Loaded daily OOS: {list(coin_dfs.keys())}')

# ── fetch 1h data for every coin (one-time pull) ───────────────────────────────
hourly = {}   # coin → 1h DataFrame

for coin, df in coin_dfs.items():
    symbol = coin + 'USDT'
    start  = str(df.index[0].date())
    end    = str(df.index[-1].date())
    klines = client.get_historical_klines(symbol, '1h', start, end)

    h = pd.DataFrame(klines, columns=[
        'Time','Open','High','Low','Close','Volume',
        'Close_time','Quote_volume','Trades','Taker_base','Taker_quote','Ignore'
    ])
    h['Time'] = pd.to_datetime(h['Time'], unit='ms', utc=True)
    for col in ['Open','High','Low','Close']:
        h[col] = h[col].astype(float)
    h = h.set_index('Time')
    hourly[coin] = h
    print(f'  {coin}: {len(h)} hourly bars  ({start} → {end})')

print('\nHourly data ready.')

---
## 2 — Set execution scenario (re-run this cell to switch, no re-fetch needed)

**Options:**
- `SCENARIO = 'close'` — baseline: fill at midnight UTC (Close[T] on signal day T)
- `SCENARIO = 10` — fill at 10am UTC on T+1, i.e. the morning after the signal (any int 0–23)
- `SCENARIO = 'worst_long'` — enter at High[T+1], exit at Low[T+1]

In [ ]:
SCENARIO = 10   # ← 'close'  |  integer hour 0-23  |  'worst_long'


def apply_scenario(coin_dfs, hourly, scenario):
    """
    Adjusts Close prices on signal days only so that entry and exit
    fill at the chosen execution price instead of midnight Close.

    The engine does: strategy_return[T] = position[T-1] * pct_change(Close)[T]
    Signal day S: position[S] changes.  effective_position[S] = position[S-1] = 0
    so Close[S] only affects bar S+1's pct_change — the execution bar.

    Replacing Close[S] with execution_price makes bar S+1 capture:
        (Close[S+1] - execution_price) / execution_price   ← entry
        (execution_price - Close[S-1]) / Close[S-1]        ← exit (last active bar)

    Both entry and exit use the same replacement: Close[S] = price at 10am[S+1]
    """
    if scenario == 'close':
        return coin_dfs

    adjusted = {}

    for coin, df in coin_dfs.items():
        d = df.copy()
        d.index = d.index.normalize()

        # signal days: any day position changes value
        signal_days = d['position'] != d['position'].shift(1).fillna(d['position'].iloc[0])

        if isinstance(scenario, int):
            # ── hourly fill at HH:00 UTC on T+1 ───────────────────────────────
            h = hourly[coin]
            prices_hh = (
                h[h.index.hour == scenario]['Close']
                .resample('1D').last()
            )
            prices_hh.index = prices_hh.index.tz_localize(None).normalize()

            # shift(-1): index T now holds the price at HH:00 on T+1
            next_exec = prices_hh.shift(-1).reindex(d.index).ffill()

            exec_close = d['Close'].copy()
            exec_close[signal_days] = next_exec[signal_days]
            d['Close'] = exec_close

        elif scenario == 'worst_long':
            # ── worst case for longs ───────────────────────────────────────────
            # entry signal day S → fill at High[S+1]  (worst buy price next day)
            # exit  signal day S → fill at Low[S+1]   (worst sell price next day)
            # High and Low are already in the pkl — no API call needed
            entry_signal = signal_days & (d['position'] != 0)   # going into position
            exit_signal  = signal_days & (d['position'] == 0)   # leaving position

            # shift(-1): index T now holds next day's High / Low
            next_high = d['High'].shift(-1)
            next_low  = d['Low'].shift(-1)

            exec_close = d['Close'].copy()
            exec_close[entry_signal] = next_high[entry_signal]
            exec_close[exit_signal]  = next_low[exit_signal]
            d['Close'] = exec_close

        adjusted[coin] = d

    return adjusted


coin_dfs_exec = apply_scenario(coin_dfs, hourly, SCENARIO)
label = f'{SCENARIO}h_UTC' if isinstance(SCENARIO, int) else SCENARIO
print(f'Scenario applied: {label}')

---
## 3 — Portfolio OOS performance

In [ ]:
SHOW_COINS = ['ETH', 'XRP', 'AVAX', 'SOL', 'BNB']   # or list(coin_dfs.keys())

metrics = plot_portfolio_oos(
    coin_dfs   = coin_dfs_exec,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},   # benchmark always uses original close
    show       = True,
    save_html  = os.path.join(
        ROOT, 'topics', 'momentum', 'results',
        f'portfolio_{label}.html'
    ),
)

print(f'\n── Scenario: {label} ──')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<20} {v:.4f}')